# yolov5 device-resident GPU training @320 → detect → show (COCO128)
Full yolov5n training device-resident (Thrust device_vector + cuBLAS), trusted host anchor
loss bridged in; saves best.pt; then `yolo detect` + displays the annotated result.
**Runtime → GPU (T4)**, Run all. (~20-30 s/epoch at 320; lower EPOCHS for a quicker demo.)


In [ ]:
!nvidia-smi -L


In [ ]:
%cd /content
!rm -rf yolov5_cpp
!git clone -q --branch main https://github.com/yomei-o/yolov5_cpp.git
%cd /content/yolov5_cpp
!wget -q https://github.com/ultralytics/yolov5/releases/download/v1.0/coco128.zip && unzip -q -o coco128.zip


### 1. Build (device + cuBLAS) and train — saves last.pt/best.pt each epoch


In [ ]:
!nvcc -x cu -O2 -std=c++17 --extended-lambda -arch=native -DUSE_CUDA -DUSE_CUBLAS -Ipure/third_party pure/dtrain_coco5.cpp -lcublas -o dtrain_coco5
EPOCHS=100  # lower (e.g. 40) for a faster demo
!./dtrain_coco5 coco128/images/train2017 320 8 $EPOCHS


### 2. Build the detector CLI (plain C++/g++) and detect with best.pt


In [ ]:
!g++ -O2 -std=c++20 -Ipure/third_party pure/yolo.cpp -o yolo
!./yolo detect --weights best.pt --source coco128/images/train2017/000000000009.jpg --imgsz 320 --conf 0.25 --out det.png


### 3. Show input + detections


In [ ]:
from IPython.display import Image, display
display(Image('coco128/images/train2017/000000000009.jpg', width=480))
display(Image('det.png', width=480))


---
COCO128 (128 imgs) overfits its training set, so detecting on a training image shows sensible boxes.
`best.pt` is a standard state_dict — loads in the yolov5 reference (torch.hub) too.
